# SetFit vs. matched baselines on BANKING77: Colab runner

Run the cells **top to bottom**. Each stage is resumable: a finished run is saved to Drive as `results/runs/<run_id>.json` and skipped next time, so after a disconnect just reconnect, run the setup cells again, and re-run the stage you were on.

Before you start: **Runtime → Change runtime type → T4 GPU**. A bigger GPU does not help this 22.7M-parameter model much and burns compute units faster.

Order: setup → smoke test → Frozen+LR → tune (seed 0) → *check curves* → select → final (8-shot first) → collect.

## 0. Setup (run every session)

In [1]:
# Folder in your Google Drive that holds experiments/ (data/ and results/ are created next to it)
ROOT = "/Users/yinzoenlouh/Desktop/Columbia/Natural Language Processing/COMS_4705_Fall_2026_Final_Project_Proposal_SetFit_BANKING77/"
CODE = f"{ROOT}/experiments"

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os, torch
assert os.path.isdir(f"{CODE}/setfit_study"), f"Upload the experiments/ folder into {ROOT} first"
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then re-run"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
%cd {CODE}

Mounted at /content/drive


AssertionError: Upload the experiments/ folder into /Users/yinzoenlouh/Desktop/Columbia/Natural Language Processing/COMS_4705_Fall_2026_Final_Project_Proposal_SetFit_BANKING77/ first

In [ ]:
# Colab normally ships transformers and scikit-learn; install only if missing.
try:
    import transformers
except ImportError:
    !pip -q install transformers
    import transformers
import sklearn
print("transformers", transformers.__version__, "| scikit-learn", sklearn.__version__)

## 1. Smoke test (about a minute)
Downloads BANKING77 and the model, checks the file hashes and the proposal's audit numbers (9,993 clean training rows, 965 validation, 9,028 pool; **it stops if anything differs**), builds the splits, then runs 10 updates of CE and SetFit plus Frozen+LR on seed 0 and prints seconds per update. A validation subset stands in for the test set here, so test labels stay untouched. Results go to `results/smoke/` and are never used in the report.

In [ ]:
!python -u -m setfit_study.run_all --root "{ROOT}" --stage smoke

## 2. Frozen+LR
Re-verifies the split files against their manifest, then fits Frozen+LR for both sizes: seed 0 on validation only, seeds 11/22/33 scored once on test (no hyperparameters to tune). It also locks the five most-confused intent pairs from seed-0 8-shot validation (`results/confusable_pairs.json`).

In [ ]:
!python -u -m setfit_study.run_all --root "{ROOT}" --stage frozen

## 3. Learning-rate tuning (seed 0, validation only)
CE and SetFit × learning rates {1e-5, 3e-5}. 8-shot first, so you have something for the milestone early.

In [ ]:
!python -u -m setfit_study.run_all --root "{ROOT}" --stage tune --shots 8

In [ ]:
!python -u -m setfit_study.run_all --root "{ROOT}" --stage tune --shots 32

### Check the curves before locking anything
Look at the CE loss panel. If CE's loss stays near **4.3** (= ln 77, chance level) while SetFit learns, the randomly initialised 77-way head is learning too slowly at encoder-sized learning rates. In that case, run the next (commented) cell to tune CE with a separate head learning rate, and report it as a change since the proposal.

In [ ]:
!python -u -m setfit_study.run_all --root "{ROOT}" --stage collect
from IPython.display import Image, display
display(Image(f"{ROOT}/results/figures/tuning_curves.png"))

In [ ]:
# Only if CE does not learn (see above):
# !python -u -m setfit_study.run_all --root "{ROOT}" --stage tune --methods ce --ce-head-lr 1e-3
# and then select CE with the same flag:
# !python -u -m setfit_study.run_all --root "{ROOT}" --stage select --methods ce --ce-head-lr 1e-3

## 4. Lock the learning rates
Picks the learning rate with the higher validation macro-F1 per method and size (ties → smaller). Once written, an entry is never changed without `--force`. Use `--shots 8` to lock 8-shot before 32-shot tuning has finished.

In [ ]:
!python -u -m setfit_study.run_all --root "{ROOT}" --stage select
!cat "{ROOT}/results/selected_hparams.json"

## 5. Final runs: seeds 11, 22, 33, each scored once on test
8-shot first. To split work across sessions or teammates, add `--seeds 22` etc.; never run the same seeds in two sessions at once.

In [ ]:
!python -u -m setfit_study.run_all --root "{ROOT}" --stage final --shots 8

In [ ]:
!python -u -m setfit_study.run_all --root "{ROOT}" --stage final --shots 32

## 6. Collect results for the report
Writes `results_log.csv` (same columns as the milestone's `results_log_template.csv`, plus extras), `summary.csv`, `paired_differences.csv`, `confusable_pairs_test.csv`, `table_main.tex` (drop-in body for Table 2) and `figures/tuning_curves.pdf`.

In [ ]:
!python -u -m setfit_study.run_all --root "{ROOT}" --stage collect
import pandas as pd
display(pd.read_csv(f"{ROOT}/results/summary.csv"))
print(open(f"{ROOT}/results/table_main.tex").read())

## When you are done
**Runtime → Disconnect and delete runtime.** A connected GPU runtime keeps using compute units even when idle. Everything you need is already in Drive under `results/`.